## 1. Imports

In [2]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

## 2. Path

In [3]:
PROJECT_DIR = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")

SPLIT_DIR = PROJECT_DIR / "splits"
MODEL_READY_DIR = PROJECT_DIR / "model_ready_no_dr"

MODEL_READY_DIR.mkdir(parents=True, exist_ok=True)

print("Split directory:", SPLIT_DIR)
print("Model-ready output directory:", MODEL_READY_DIR)

Split directory: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/splits
Model-ready output directory: /Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_ready_no_dr


## 3. Loading the data splits created from EDA

In [4]:
train_path = SPLIT_DIR / "train_model_ready_selected_features.csv"
val_path = SPLIT_DIR / "val_model_ready_selected_features.csv"
test_path = SPLIT_DIR / "test_model_ready_selected_features.csv"

for path in [train_path, val_path, test_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

train_df = pd.read_csv(train_path, low_memory=False)
val_df = pd.read_csv(val_path, low_memory=False)
test_df = pd.read_csv(test_path, low_memory=False)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

Train shape: (694549, 19)
Validation shape: (138749, 19)
Test shape: (138524, 19)

Train columns:
['log_id', 'dataset_label', 'battery_status__cell_count', 'battery_status__current_a', 'battery_status__current_filtered_a', 'battery_status__discharged_mah', 'battery_status__remaining', 'battery_status__scale', 'battery_status__voltage_cell_v[0]', 'battery_status__voltage_cell_v[1]', 'battery_status__voltage_cell_v[2]', 'battery_status__voltage_cell_v[3]', 'battery_status__voltage_filtered_v', 'battery_status__voltage_v', 'cumulative_energy_wh', 'energy_increment_wh', 'power_w', 'time_s', 'target_energy_next_60s_wh']


## 4. feature colomn and target colomn 

In [5]:
target_col = "target_energy_next_60s_wh"

required_cols = ["log_id", "dataset_label", target_col]

for col in required_cols:
    if col not in train_df.columns:
        raise ValueError(f"Required column missing from train_df: {col}")

non_feature_cols = [
    "log_id",
    "dataset_label",
    target_col
]

feature_cols = [
    col for col in train_df.columns
    if col not in non_feature_cols
    and pd.api.types.is_numeric_dtype(train_df[col])
]

print("Feature count before duplicate removal:", len(feature_cols))
print(feature_cols)

Feature count before duplicate removal: 16
['battery_status__cell_count', 'battery_status__current_a', 'battery_status__current_filtered_a', 'battery_status__discharged_mah', 'battery_status__remaining', 'battery_status__scale', 'battery_status__voltage_cell_v[0]', 'battery_status__voltage_cell_v[1]', 'battery_status__voltage_cell_v[2]', 'battery_status__voltage_cell_v[3]', 'battery_status__voltage_filtered_v', 'battery_status__voltage_v', 'cumulative_energy_wh', 'energy_increment_wh', 'power_w', 'time_s']


## 5.  duplicate feature removal  
because power_w already captures the same energy-use signal at 10 Hz

In [ ]:
if "power_w" in feature_cols and "energy_increment_wh" in feature_cols:
    feature_cols.remove("energy_increment_wh")
    print("Removed energy_increment_wh .")

print("Final feature count:", len(feature_cols))
print(feature_cols)

Removed energy_increment_wh because power_w already captures the same energy-use signal at 10 Hz.
Final feature count: 15
['battery_status__cell_count', 'battery_status__current_a', 'battery_status__current_filtered_a', 'battery_status__discharged_mah', 'battery_status__remaining', 'battery_status__scale', 'battery_status__voltage_cell_v[0]', 'battery_status__voltage_cell_v[1]', 'battery_status__voltage_cell_v[2]', 'battery_status__voltage_cell_v[3]', 'battery_status__voltage_filtered_v', 'battery_status__voltage_v', 'cumulative_energy_wh', 'power_w', 'time_s']


## 6. Flight overlap check

In [7]:
train_ids = set(train_df["log_id"].astype(str).unique())
val_ids = set(val_df["log_id"].astype(str).unique())
test_ids = set(test_df["log_id"].astype(str).unique())

print("Train-Val overlap:", len(train_ids & val_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Val-Test overlap:", len(val_ids & test_ids))

assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

print("No flight-level leakage found.")

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0
No flight-level leakage found.


## 7. X and y created

In [8]:
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df[target_col].copy()
y_val = val_df[target_col].copy()
y_test = test_df[target_col].copy()

# Replace infinite values with NaN before imputation
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (694549, 15)
X_val: (138749, 15)
X_test: (138524, 15)


## 8. Missing-value check before imputation

In [9]:
missing_report = pd.DataFrame({
    "feature": feature_cols,
    "train_missing_%": X_train.isna().mean().values * 100,
    "val_missing_%": X_val.isna().mean().values * 100,
    "test_missing_%": X_test.isna().mean().values * 100
}).sort_values("train_missing_%", ascending=False)

display(missing_report)

missing_report.to_csv(
    MODEL_READY_DIR / "missing_value_report_before_imputation_no_dr.csv",
    index=False
)

,feature,train_missing_%,val_missing_%,test_missing_%
6,battery_status__voltage_cell_v[0],76.07685,75.316579,70.920563
7,battery_status__voltage_cell_v[1],76.07685,75.316579,70.920563
8,battery_status__voltage_cell_v[2],76.07685,75.316579,70.920563
9,battery_status__voltage_cell_v[3],76.07685,75.316579,70.920563
0,battery_status__cell_count,0.00000,0.000000,0.000000
1,battery_status__current_a,0.00000,0.000000,0.000000
2,battery_status__current_filtered_a,0.00000,0.000000,0.000000
3,battery_status__discharged_mah,0.00000,0.000000,0.000000
4,battery_status__remaining,0.00000,0.000000,0.000000
5,battery_status__scale,0.00000,0.000000,0.000000


## 9. Median imputation

In [10]:
imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

print("Missing values after median imputation:")
print("Train:", np.isnan(X_train_imputed).sum())
print("Validation:", np.isnan(X_val_imputed).sum())
print("Test:", np.isnan(X_test_imputed).sum())

Missing values after median imputation:
Train: 0
Validation: 0
Test: 0


## 10. imputed-only datasets creation

In [12]:
X_train_imputed_df = pd.DataFrame(X_train_imputed, columns=feature_cols)
X_val_imputed_df = pd.DataFrame(X_val_imputed, columns=feature_cols)
X_test_imputed_df = pd.DataFrame(X_test_imputed, columns=feature_cols)

train_no_dr_imputed = pd.concat([
    train_df[["log_id", "dataset_label"]].reset_index(drop=True),
    X_train_imputed_df.reset_index(drop=True),
    y_train.reset_index(drop=True).rename(target_col)
], axis=1)

val_no_dr_imputed = pd.concat([
    val_df[["log_id", "dataset_label"]].reset_index(drop=True),
    X_val_imputed_df.reset_index(drop=True),
    y_val.reset_index(drop=True).rename(target_col)
], axis=1)

test_no_dr_imputed = pd.concat([
    test_df[["log_id", "dataset_label"]].reset_index(drop=True),
    X_test_imputed_df.reset_index(drop=True),
    y_test.reset_index(drop=True).rename(target_col)
], axis=1)

print("Imputed train:", train_no_dr_imputed.shape)
print("Imputed validation:", val_no_dr_imputed.shape)
print("Imputed test:", test_no_dr_imputed.shape)

Imputed train: (694549, 18)
Imputed validation: (138749, 18)
Imputed test: (138524, 18)


## 11. Standardisation / scaling

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Scaled shapes:")
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

Scaled shapes:
Train: (694549, 15)
Validation: (138749, 15)
Test: (138524, 15)


In [14]:
scaled_check = pd.DataFrame(X_train_scaled, columns=feature_cols)

scale_summary = pd.DataFrame({
    "feature": feature_cols,
    "scaled_train_mean": scaled_check.mean().values,
    "scaled_train_std": scaled_check.std().values
})

display(scale_summary)

scale_summary.to_csv(
    MODEL_READY_DIR / "scaling_check_train_no_dr.csv",
    index=False
)

,feature,scaled_train_mean,scaled_train_std
0,battery_status__cell_count,0.000000e+00,1.000001
1,battery_status__current_a,1.990402e-16,1.000001
2,battery_status__current_filtered_a,9.428222e-17,1.000001
3,battery_status__discharged_mah,1.466612e-16,1.000001
4,battery_status__remaining,1.335665e-16,1.000001
5,battery_status__scale,1.539943e-15,1.000001
6,battery_status__voltage_cell_v[0],2.095160e-17,1.000001
7,battery_status__voltage_cell_v[1],2.095160e-17,1.000001
8,battery_status__voltage_cell_v[2],2.095160e-17,1.000001
9,battery_status__voltage_cell_v[3],2.095160e-17,1.000001


## 12. Scaled model ready datasets

In [15]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=feature_cols)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_cols)

train_no_dr_scaled = pd.concat([
    train_df[["log_id", "dataset_label"]].reset_index(drop=True),
    X_train_scaled_df.reset_index(drop=True),
    y_train.reset_index(drop=True).rename(target_col)
], axis=1)

val_no_dr_scaled = pd.concat([
    val_df[["log_id", "dataset_label"]].reset_index(drop=True),
    X_val_scaled_df.reset_index(drop=True),
    y_val.reset_index(drop=True).rename(target_col)
], axis=1)

test_no_dr_scaled = pd.concat([
    test_df[["log_id", "dataset_label"]].reset_index(drop=True),
    X_test_scaled_df.reset_index(drop=True),
    y_test.reset_index(drop=True).rename(target_col)
], axis=1)

print("Scaled train:", train_no_dr_scaled.shape)
print("Scaled validation:", val_no_dr_scaled.shape)
print("Scaled test:", test_no_dr_scaled.shape)

display(train_no_dr_scaled.head())

Scaled train: (694549, 18)
Scaled validation: (138749, 18)
Scaled test: (138524, 18)


,log_id,dataset_label,battery_status__cell_count,battery_status__current_a,battery_status__current_filtered_a,battery_status__discharged_mah,battery_status__remaining,battery_status__scale,battery_status__voltage_cell_v[0],battery_status__voltage_cell_v[1],battery_status__voltage_cell_v[2],battery_status__voltage_cell_v[3],battery_status__voltage_filtered_v,battery_status__voltage_v,cumulative_energy_wh,power_w,time_s,target_energy_next_60s_wh
0,00209ad8-f7c4-4d6f-839c-af19d8203e9b,normal,-0.411957,-0.884614,-0.887438,-0.654732,0.530812,-0.492240,-0.241039,-0.241039,-0.241039,-0.241039,-0.418736,-0.414391,-0.343775,-0.617679,-1.091536,0.001262
1,00209ad8-f7c4-4d6f-839c-af19d8203e9b,normal,-0.411957,-0.884614,-0.887438,-0.654732,0.530812,-0.492240,-0.241039,-0.241039,-0.241039,-0.241039,-0.418736,-0.414391,-0.343775,-0.617679,-1.091260,0.001262
2,00209ad8-f7c4-4d6f-839c-af19d8203e9b,normal,-0.411957,-0.884614,-0.887438,-0.654732,0.530812,-0.492240,-0.241039,-0.241039,-0.241039,-0.241039,-0.418736,-0.414391,-0.343775,-0.617679,-1.090985,0.001262
3,00209ad8-f7c4-4d6f-839c-af19d8203e9b,normal,-0.411957,-0.884614,-0.887438,-0.654732,0.530812,-0.492240,-0.241039,-0.241039,-0.241039,-0.241039,-0.418736,-0.414391,-0.343775,-0.617679,-1.090709,0.001262
4,00209ad8-f7c4-4d6f-839c-af19d8203e9b,normal,-0.411957,-0.838245,-0.891093,-0.654728,0.321380,-0.285151,-0.241039,-0.241039,-0.241039,-0.241039,-0.419166,-0.418824,-0.343772,-0.595105,-1.090433,0.001109


## 13. Final dataset check

In [16]:
def check_final_dataset(name, data):
    print(f"\n{name}")
    print("-" * 40)
    print("Shape:", data.shape)
    print("Missing values:", data.isna().sum().sum())
    print("Unique flights:", data["log_id"].nunique())
    print("Target missing:", data[target_col].isna().sum())
    print("Label distribution:")
    print(data["dataset_label"].value_counts())

check_final_dataset("Train imputed", train_no_dr_imputed)
check_final_dataset("Validation imputed", val_no_dr_imputed)
check_final_dataset("Test imputed", test_no_dr_imputed)

check_final_dataset("Train scaled", train_no_dr_scaled)
check_final_dataset("Validation scaled", val_no_dr_scaled)
check_final_dataset("Test scaled", test_no_dr_scaled)


Train imputed
----------------------------------------
Shape: (694549, 18)
Missing values: 0
Unique flights: 117
Target missing: 0
Label distribution:
dataset_label
normal      497250
abnormal    197299
Name: count, dtype: int64

Validation imputed
----------------------------------------
Shape: (138749, 18)
Missing values: 0
Unique flights: 25
Target missing: 0
Label distribution:
dataset_label
normal      92724
abnormal    46025
Name: count, dtype: int64

Test imputed
----------------------------------------
Shape: (138524, 18)
Missing values: 0
Unique flights: 26
Target missing: 0
Label distribution:
dataset_label
normal      100406
abnormal     38118
Name: count, dtype: int64

Train scaled
----------------------------------------
Shape: (694549, 18)
Missing values: 0
Unique flights: 117
Target missing: 0
Label distribution:
dataset_label
normal      497250
abnormal    197299
Name: count, dtype: int64

Validation scaled
----------------------------------------
Shape: (138749, 18)
M

## 14. File saving

In [17]:
train_no_dr_imputed.to_csv(MODEL_READY_DIR / "train_no_dr_imputed.csv", index=False)
val_no_dr_imputed.to_csv(MODEL_READY_DIR / "val_no_dr_imputed.csv", index=False)
test_no_dr_imputed.to_csv(MODEL_READY_DIR / "test_no_dr_imputed.csv", index=False)

train_no_dr_scaled.to_csv(MODEL_READY_DIR / "train_no_dr_scaled.csv", index=False)
val_no_dr_scaled.to_csv(MODEL_READY_DIR / "val_no_dr_scaled.csv", index=False)
test_no_dr_scaled.to_csv(MODEL_READY_DIR / "test_no_dr_scaled.csv", index=False)

joblib.dump(imputer, MODEL_READY_DIR / "median_imputer_no_dr.joblib")
joblib.dump(scaler, MODEL_READY_DIR / "standard_scaler_no_dr.joblib")

with open(MODEL_READY_DIR / "final_feature_columns_no_dr.json", "w") as f:
    json.dump(feature_cols, f, indent=4)

print("Saved all no-dimensionality-reduction model-ready files to:")
print(MODEL_READY_DIR)

Saved all no-dimensionality-reduction model-ready files to:
/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project/model_ready_no_dr


#For Random Forest / XGBoost:
train_no_dr_imputed.csv
val_no_dr_imputed.csv
test_no_dr_imputed.csv

#For Linear Regression / LSTM / Autoencoder:
train_no_dr_scaled.csv
val_no_dr_scaled.csv
test_no_dr_scaled.csv

Two versions of the dataset were prepared. An imputed but unscaled dataset was retained for Random Forest and XGBoost because tree-based algorithms are generally insensitive to feature scaling. A standardised version of the dataset was also created for Linear Regression, LSTM, Bi-LSTM and Autoencoder models, as these methods typically benefit from inputs being on a similar numerical scale.